In [16]:
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

In [34]:
df = pd.read_csv('revised_recipes.csv')
df['web_surfix'] = (df['name'].astype(str) + ' ' + df['id'].astype(str)).str.replace(' ', '-')
#add columns to df for scraped data
df['scraped_info'] = None
df['amount_list'] = None
df['unit_list'] = None
df['full_text_list'] = None


In [25]:
def scrape_recipe(web_surfix):
    
    url = f'https://www.food.com/recipe/{web_surfix}/'
    # Code to scrape the recipe from the URL
    # This is a placeholder for the actual scraping logic

    # Initialize your driver (adjust for Chrome/Firefox as needed)
    driver = webdriver.Chrome()
    driver.get(url)
    time.sleep(1)  # Optional: wait a moment before starting the scraping

    amount_list = []
    unit_list = []
    full_text_list = []

    # 1. Locate the main ingredient list container
    try:
        ingredient_list = driver.find_element(By.CLASS_NAME, "ingredient-list")
        # Alternatively, use the full class if needed:
        # ingredient_list = driver.find_element(By.CSS_SELECTOR, ".ingredient-list.svelte-ik1ga1")
        
        # 2. Find all list items (li) inside the container
        ingredients = ingredient_list.find_elements(By.TAG_NAME, "li")

        for item in ingredients:
            try:
                # Extract the raw quantity string
                qty_el = item.find_element(By.CLASS_NAME, "ingredient-quantity")
                quantity = qty_el.text.strip()
                
                # Extract the text description string
                text_el = item.find_element(By.CLASS_NAME, "ingredient-text")
                full_text = text_el.text.strip()
                
                # 3. Regex logic to capture the unit
                # This looks inside parentheses if they exist, or grabs the first word
                unit_match = re.search(r'^\((.*?)\)|^(\w+)', full_text)
                
                unit = ""
                if unit_match:
                    # Group 1 handles parentheses like "(16 ounce) can" -> "16 ounce"
                    # Group 2 handles normal first words like "teaspoon salt" -> "teaspoon"
                    unit = unit_match.group(1) if unit_match.group(1) else unit_match.group(2)
                
                amount_list.append(quantity)
                unit_list.append(unit)
                full_text_list.append(full_text)
            except Exception as e:
                # Handles empty list items or structure variations gracefully
                continue

    finally:
        time.sleep(1)  # Optional: wait a moment before closing the driver
        driver.quit()
    
      # Ensure the driver has fully closed before proceeding

    return amount_list, unit_list, full_text_list
    

In [35]:
df['amount_list'], df['unit_list'], df['full_text_list'] = zip(*df['web_surfix'].apply(scrape_recipe))

In [36]:
df.to_csv('scraped_recipe_ingredients.csv', index=False)